<a href="https://colab.research.google.com/github/maiarcaio/lab-extractive-question-answering/blob/main/lab-extractive-question-answering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LAB | Extractive Question Answering

<div style="border:1px solid #ccc; border-radius:6px; text-align: center;">

<br>

**Run this notebook in [Google Colab](https://colab.research.google.com/) for GPU acceleration.**

<br>

</div>

This notebook demonstrates how Pinecone helps you build an extractive question-answering application. To build an extractive question-answering system, we need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A reader model to extract answers

We will use the SQuAD dataset, which consists of **questions** and **context** paragraphs containing question **answers**. We generate embeddings for the context passages using the retriever, index them in the vector database, and query with semantic search to retrieve the top k most relevant contexts containing potential answers to our question. We then use the reader model to extract the answers from the returned contexts.

Let's get started by installing the packages needed for notebook to run:

In [1]:
# Import PyTorch to check GPU availability.
import torch

# Show which PyTorch version is installed.
print("PyTorch:", torch.__version__)

# Check whether CUDA can access the NVIDIA GPU.
print("CUDA available:", torch.cuda.is_available())

# Identify the GPU if CUDA is available.
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Check whether the NVIDIA GPU is attached to this runtime.
!nvidia-smi

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
Tue Oct 27 14:08:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |               

# Install Dependencies

In [1]:
# Check whether the required libraries import successfully.
import torch
import transformers
import datasets
import pinecone
import sentence_transformers
import huggingface_hub

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("CUDA available:", torch.cuda.is_available())

# Confirm that the Pinecone client can be imported.
from pinecone import Pinecone
print("Pinecone import: OK")

PyTorch: 2.11.0+cu130
Transformers: 4.46.3
Datasets: 4.5.0
Sentence Transformers: 3.3.1
Hugging Face Hub: 0.26.5
CUDA available: True
Pinecone import: OK


In [ ]:
# Restart Python so the newly installed package versions are loaded.
import os

# Terminate the current process; Colab should reconnect automatically.
os.kill(os.getpid(), 9)

# Load Dataset

Now let's load the SQUAD dataset from the HuggingFace Model Hub. We load the dataset into a pandas dataframe and filter the title, question, and context columns, and we drop any duplicate context passages.

In [1]:
from datasets import load_dataset

# Download the SQuAD training dataset from Hugging Face.
# Convert it into a pandas DataFrame for easier exploration.
df = load_dataset("rajpurkar/squad", split="train").to_pandas()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

# Pinecone Setup

Pinecone is a managed vector database that lets us store context embeddings and run fast semantic search over them. In this section, we set up the connection to Pinecone using an API key and create the index that will hold our context embeddings.

In [2]:
# Keep only the columns needed for semantic search.
# "title" identifies the article; "context" contains the source passage.
df = df[["title", "context"]]

# Remove duplicate passages.
# SQuAD can contain multiple questions referring to the same context.
df = df.drop_duplicates(subset="context")

# Display the cleaned dataset.
display(df.head())

# Check how many unique context passages remain.
print("Unique passages:", len(df))

,title,context
0,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha..."
5,University_of_Notre_Dame,"As at most other universities, Notre Dame's st..."
10,University_of_Notre_Dame,The university is the major seat of the Congre...
15,University_of_Notre_Dame,The College of Engineering was established in ...
20,University_of_Notre_Dame,All of Notre Dame's undergraduate students are...


Unique passages: 18891


In [3]:
import os
import getpass
from pinecone import Pinecone

#
# For this notebook, you'll need a Pinecone API key,
# to get one, sign up for free at https://app.pinecone.io/,
# then go to "API Keys" and copy the default key (or create a new one).
#
# Once you have your Pinecone API key,
# load it from Colab's "Secrets" manager
# (the key icon in the left sidebar).
# Add a secret named PINECONE_API_KEY there and enable notebook access.
#
# Note: Colab Secrets rely on a UI popup to grant access, which only works when
# running in the Colab UI itself. If you're running this notebook through
# another editor (e.g. VS Code's Colab extension), that popup isn't available,
# so we fall back to an environment variable or a secure manual prompt instead.
api_key = os.environ.get('PINECONE_API_KEY')


if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('PINECONE_API_KEY')
    except Exception:
        api_key = None

if not api_key:
    api_key = getpass.getpass('Enter your Pinecone API key: ')



<br>

Once we have the API key, we can initialize the connection:

In [4]:
# Import the Pinecone client and the serverless index configuration.
from pinecone import Pinecone, ServerlessSpec

# Configure the cloud provider and region for our index.
spec = ServerlessSpec(
    cloud="aws",
    region="us-east-1"
)

# Initialize the Pinecone client using the securely loaded API key.
pc = Pinecone(api_key=api_key)

# Confirm that we can access the account's indexes.
print("Existing indexes:", pc.list_indexes().names())

Existing indexes: []


Now we create a new index called "question-answering" — we can name the index anything we want. We specify the metric type as "cosine" and dimension as 384 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 384-dimension vectors.

In [5]:
# Give our vector index a descriptive name.
index_name = "extractive-question-answering"

# Check whether the index already exists to avoid creating it twice.
if index_name not in pc.list_indexes().names():

    # Create an index to store vectors containing 384 numbers each.
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=spec
    )

# Connect to our index so we can insert and search vectors.
index = pc.Index(index_name)

# Check whether the index currently contains any vectors.
print(index.describe_index_stats())

{'dimension': 384,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}


# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all context passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will generate embeddings in a way that the questions and context passages containing answers to our questions are nearby in the vector space. We can use cosine similarity to calculate the similarity between the query and context embeddings to find the context passages that contain potential answers to our question.

We will use a SentenceTransformer model named ``multi-qa-MiniLM-L6-cos-v1`` designed for semantic search and trained on 215M (question, answer) pairs from diverse sources as our retriever.

In [8]:
# Import PyTorch to check whether a GPU is available.
import torch

# Import SentenceTransformer to convert text into embeddings.
from sentence_transformers import SentenceTransformer

# Use the GPU if available; otherwise use the CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the pretrained embedding model.
retriever = SentenceTransformer(
    "sentence-transformers/multi-qa-MiniLM-L6-cos-v1",
    device=device
)

# Confirm that the model loaded successfully.
print("Device:", device)
print("Embedding dimensions:", retriever.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Device: cuda
Embedding dimensions: 384


# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, context passage, etc.

In [9]:
# Import tqdm to display a progress bar.
from tqdm.auto import tqdm

# Process 64 passages at a time.
batch_size = 64

# Loop through the dataset in steps of 64.
for i in tqdm(range(0, len(df), batch_size)):

    # Find the ending position without exceeding the dataset length.
    i_end = min(i + batch_size, len(df))

    # Select the passages in the current batch.
    batch = df.iloc[i:i_end]

    # Convert each context passage into a 384-dimensional embedding.
    emb = retriever.encode(batch["context"].tolist())

    # Create metadata containing each passage's title and original text.
    meta = batch[["title", "context"]].to_dict(orient="records")

    # Create a unique ID for every passage using its DataFrame index.
    ids = [str(idx) for idx in batch.index]

    # Combine IDs, embeddings, and metadata into Pinecone records.
    to_upsert = [
        {
            "id": ids[j],
            "values": emb[j].tolist(),
            "metadata": meta[j]
        }
        for j in range(len(ids))
    ]

    # Insert or update these records in our Pinecone index.
    index.upsert(vectors=to_upsert)

# Check how many vectors Pinecone reports after uploading.
print(index.describe_index_stats())

  0%|          | 0/296 [00:00<?, ?it/s]

{'dimension': 384,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 18891}},
 'total_vector_count': 18891}


In [10]:
# Select only two passages for a small test.
test_batch = df.iloc[:2]

# Generate embeddings for these passages.
test_embeddings = retriever.encode(test_batch["context"].tolist())

# Inspect their shape.
print("Embedding shape:", test_embeddings.shape)

# Check the first passage's title.
print("First title:", test_batch.iloc[0]["title"])

Embedding shape: (2, 384)
First title: University_of_Notre_Dame


# Initialize Reader

We use the `deepset/electra-base-squad2` model from the HuggingFace model hub as our reader model. We load this model into a "question-answering" pipeline from HuggingFace transformers and feed it our questions and context passages individually. The model gives a prediction for each context we pass through the pipeline.

In [11]:
# Import Hugging Face's pipeline function.
from transformers import pipeline

# Select a pretrained model designed for extractive question answering.
model_name = "deepset/electra-base-squad2"

# Use GPU 0 if CUDA is available; otherwise use CPU.
device_id = 0 if torch.cuda.is_available() else -1

# Create a question-answering pipeline using the pretrained model.
reader = pipeline(
    task="question-answering",
    model=model_name,
    tokenizer=model_name,
    device=device_id
)

# Confirm that the pipeline has been initialized.
print("Reader model:", model_name)
print("Device:", reader.device)

config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Reader model: deepset/electra-base-squad2
Device: cuda:0


Now all the components we need are ready. Let's write some helper functions to execute our queries. The `get_context` function retrieves the context embeddings containing answers to our question from the Pinecone index, and the `extract_answer` function extracts the answers from these context passages.

In [17]:
# Retrieve the most relevant passages for a question.
def get_context(question, top_k):

    # Convert the question into a 384-dimensional embedding.
    xq = retriever.encode(question).tolist()

    # Search Pinecone for the top_k most similar passages.
    # include_metadata=True returns the original text with each result.
    xc = index.query(
        vector=xq,
        top_k=top_k,
        include_metadata=True
    )

    # Extract the readable context from each matching record.
    c = [
        match["metadata"]["context"]
        for match in xc["matches"]
    ]

    # Return a list of retrieved passages.
    return c

In [18]:
from pprint import pprint

# Extract candidate answers from the retrieved passages.
def extract_answer(question, context):

    # Store predictions from the reader model.
    results = []

    # Process each context passage individually.
    for c in context:

        # Ask ELECTRA to extract an answer from this passage.
        answer = reader(question=question, context=c)

        # Keep the source passage alongside the prediction.
        answer["context"] = c

        # Add the result to our list.
        results.append(answer)

    # Rank predictions by the reader model's confidence score.
    sorted_results = sorted(
        results,
        key=lambda x: x["score"],
        reverse=True
    )

    # Display predictions in a readable format.
    pprint(sorted_results)

    # Return the actual list so other code can use it.
    return sorted_results

In [19]:
question = "How much oil is Egypt producing in a day?"
context = get_context(question, top_k = 1)
context

['Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of natural gas (in 2013), which makes Egypt as the largest oil producer not member of the Organization of the Petroleum Exporting Countries (OPEC) and the second-largest dry natural gas producer in Africa. In 2013, Egypt was the largest consumer of oil and natural gas in Africa, as more than 20% of total oil consumption and more than 40% of total dry natural gas consumption in Africa. Also, Egypt possesses the largest oil refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is currently planning to build its first nuclear power plant in El Dabaa city, northern Egypt.']

As we can see, the retiever is working fine and gets us the context passage that contains the answer to our question. Now let's use the reader to extract the exact answer from the context passage.

In [20]:
extract_answer(question, context)

[{'answer': '691,000 bbl/d',
  'context': 'Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of '
             'natural gas (in 2013), which makes Egypt as the largest oil '
             'producer not member of the Organization of the Petroleum '
             'Exporting Countries (OPEC) and the second-largest dry natural '
             'gas producer in Africa. In 2013, Egypt was the largest consumer '
             'of oil and natural gas in Africa, as more than 20% of total oil '
             'consumption and more than 40% of total dry natural gas '
             'consumption in Africa. Also, Egypt possesses the largest oil '
             'refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is '
             'currently planning to build its first nuclear power plant in El '
             'Dabaa city, northern Egypt.',
  'end': 33,
  'score': 0.9999852180480957,
  'start': 20}]


[{'score': 0.9999852180480957,
  'start': 20,
  'end': 33,
  'answer': '691,000 bbl/d',
  'context': 'Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of natural gas (in 2013), which makes Egypt as the largest oil producer not member of the Organization of the Petroleum Exporting Countries (OPEC) and the second-largest dry natural gas producer in Africa. In 2013, Egypt was the largest consumer of oil and natural gas in Africa, as more than 20% of total oil consumption and more than 40% of total dry natural gas consumption in Africa. Also, Egypt possesses the largest oil refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is currently planning to build its first nuclear power plant in El Dabaa city, northern Egypt.'}]

The reader model predicted with 99% accuracy the correct answer *691,000 bbl/d* as seen from the context passage. Let's run few more queries.

In [21]:
question = "What are the first names of the men that invented youtube?"
context = get_context(question, top_k=1)
extract_answer(question, context)

[{'answer': 'Hurley and Chen',
  'context': 'According to a story that has often been repeated in the media, '
             'Hurley and Chen developed the idea for YouTube during the early '
             'months of 2005, after they had experienced difficulty sharing '
             "videos that had been shot at a dinner party at Chen's apartment "
             'in San Francisco. Karim did not attend the party and denied that '
             'it had occurred, but Chen commented that the idea that YouTube '
             'was founded after a dinner party "was probably very strengthened '
             'by marketing ideas around creating a story that was very '
             'digestible".',
  'end': 79,
  'score': 0.9999276399612427,
  'start': 64}]


[{'score': 0.9999276399612427,
  'start': 64,
  'end': 79,
  'answer': 'Hurley and Chen',
  'context': 'According to a story that has often been repeated in the media, Hurley and Chen developed the idea for YouTube during the early months of 2005, after they had experienced difficulty sharing videos that had been shot at a dinner party at Chen\'s apartment in San Francisco. Karim did not attend the party and denied that it had occurred, but Chen commented that the idea that YouTube was founded after a dinner party "was probably very strengthened by marketing ideas around creating a story that was very digestible".'}]

In [22]:
question = "What is Albert Eistein famous for?"
context = get_context(question, top_k=1)
extract_answer(question, context)

[{'answer': 'his theories of special relativity and general relativity',
  'context': 'Albert Einstein is known for his theories of special relativity '
             'and general relativity. He also made important contributions to '
             'statistical mechanics, especially his mathematical treatment of '
             'Brownian motion, his resolution of the paradox of specific '
             'heats, and his connection of fluctuations and dissipation. '
             'Despite his reservations about its interpretation, Einstein also '
             'made contributions to quantum mechanics and, indirectly, quantum '
             'field theory, primarily through his theoretical studies of the '
             'photon.',
  'end': 86,
  'score': 0.9500371217727661,
  'start': 29}]


[{'score': 0.9500371217727661,
  'start': 29,
  'end': 86,
  'answer': 'his theories of special relativity and general relativity',
  'context': 'Albert Einstein is known for his theories of special relativity and general relativity. He also made important contributions to statistical mechanics, especially his mathematical treatment of Brownian motion, his resolution of the paradox of specific heats, and his connection of fluctuations and dissipation. Despite his reservations about its interpretation, Einstein also made contributions to quantum mechanics and, indirectly, quantum field theory, primarily through his theoretical studies of the photon.'}]

Let's run another question. This time for top 3 context passages from the retriever.

In [23]:
question = "Who was the first person to step foot on the moon?"
context = get_context(question, top_k=3)
extract_answer(question, context)

[{'answer': 'Armstrong',
  'context': 'The trip to the Moon took just over three days. After achieving '
             'orbit, Armstrong and Aldrin transferred into the Lunar Module, '
             'named Eagle, and after a landing gear inspection by Collins '
             'remaining in the Command/Service Module Columbia, began their '
             'descent. After overcoming several computer overload alarms '
             'caused by an antenna switch left in the wrong position, and a '
             'slight downrange error, Armstrong took over manual flight '
             'control at about 180 meters (590 ft), and guided the Lunar '
             'Module to a safe landing spot at 20:18:04 UTC, July 20, 1969 '
             '(3:17:04 pm CDT). The first humans on the Moon would wait '
             'another six hours before they ventured out of their craft. At '
             '02:56 UTC, July 21 (9:56 pm CDT July 20), Armstrong became the '
             'first human to set foot on the Moon.',

[{'score': 0.9998037815093994,
  'start': 71,
  'end': 80,
  'answer': 'Armstrong',
  'context': 'The trip to the Moon took just over three days. After achieving orbit, Armstrong and Aldrin transferred into the Lunar Module, named Eagle, and after a landing gear inspection by Collins remaining in the Command/Service Module Columbia, began their descent. After overcoming several computer overload alarms caused by an antenna switch left in the wrong position, and a slight downrange error, Armstrong took over manual flight control at about 180 meters (590 ft), and guided the Lunar Module to a safe landing spot at 20:18:04 UTC, July 20, 1969 (3:17:04 pm CDT). The first humans on the Moon would wait another six hours before they ventured out of their craft. At 02:56 UTC, July 21 (9:56 pm CDT July 20), Armstrong became the first human to set foot on the Moon.'},
 {'score': 0.695867121219635,
  'start': 240,
  'end': 246,
  'answer': 'Aldrin',
  'context': 'The first step was witnessed by at 

<br>

### 🚀 Challenge

Try adding a few more questions of your own in new cells below (reusing `get_context` and `extract_answer`). Experiment with different `top_k` values and observe:

- Does the retriever always return a context that actually contains the answer?
- How does the reader's confidence score change with harder or more ambiguous questions?
- What happens with questions that have no good answer in the dataset?

In [24]:
# Experiment 1: Compare retrieval with different top_k values.
question = "Who was the first person to walk on the Moon?"

for k in [1, 3, 5]:

    # Retrieve k potentially relevant passages from Pinecone.
    contexts = get_context(question, top_k=k)

    # Extract possible answers using the ELECTRA reader.
    answers = [
        reader(question=question, context=passage)
        for passage in contexts
    ]

    # Rank predictions by the reader's confidence score.
    answers = sorted(
        answers,
        key=lambda answer: answer["score"],
        reverse=True
    )

    # Display the highest-ranked prediction for this experiment.
    print(f"\nTop-k: {k}")

    if answers:
        print("Best answer:", answers[0]["answer"])
        print("Confidence:", round(answers[0]["score"], 4))
    else:
        print("No passages retrieved.")


Top-k: 1
Best answer: Armstrong
Confidence: 1.0

Top-k: 3
Best answer: Armstrong
Confidence: 1.0


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



Top-k: 5
Best answer: Armstrong
Confidence: 1.0


In [25]:
# Experiment 2: Test how the model handles an unsupported question.
question = "Who will win the FIFA World Cup in 2030?"

# Retrieve three passages despite the question requiring future knowledge.
contexts = get_context(question, top_k=3)

# Try extracting an answer from those passages.
answers = extract_answer(question, contexts)

[{'answer': 'The winners of each continental competition contest the FIFA Club '
            'World Cup.',
  'context': 'After the World Cup, the most important international football '
             'competitions are the continental championships, which are '
             'organised by each continental confederation and contested '
             'between national teams. These are the European Championship '
             '(UEFA), the Copa América (CONMEBOL), African Cup of Nations '
             '(CAF), the Asian Cup (AFC), the CONCACAF Gold Cup (CONCACAF) and '
             'the OFC Nations Cup (OFC). The FIFA Confederations Cup is '
             'contested by the winners of all six continental championships, '
             'the current FIFA World Cup champions and the country which is '
             'hosting the Confederations Cup. This is generally regarded as a '
             'warm-up tournament for the upcoming FIFA World Cup and does not '
             'carry the same prestige as t

<br>

---

### Clean Up

Now that we're done experimenting, let's delete the Pinecone index we created. This frees up the resources it was using (and, when using a paid plan, avoids being billed for an index we no longer need).

In [26]:
# Delete the Pinecone index after completing the lab.
# WARNING: This permanently removes all vectors stored in this index.
pc.delete_index(index_name)

# Confirm that the index no longer exists.
print("Remaining indexes:", pc.list_indexes().names())

Remaining indexes: []
